In [1]:
import numpy as np
import pandas as pd

from gensim.models import Word2Vec

import nltk
from nltk.tokenize import RegexpTokenizer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

from sklearn.linear_model import LogisticRegression

# Fix: ensure required NLTK resources exist in Kaggle environment (prevents LookupError).
for pkg in ["stopwords", "wordnet", "omw-1.4"]:
    try:
        nltk.data.find(f"corpora/{pkg}")
    except LookupError:
        nltk.download(pkg, quiet=True)

alpha_tokenizer = RegexpTokenizer(r"[A-Za-z]\w+")
lemmatizer = WordNetLemmatizer()
stop = set(stopwords.words("english"))



In [2]:
# Fix: use the correct Kaggle input path for this environment.
train = pd.read_csv("/kaggle/input/train.csv")
test = pd.read_csv("/kaggle/input/test.csv")



In [3]:
data = [
    [
        lemmatizer.lemmatize(word.lower())
        for word in alpha_tokenizer.tokenize(sent)
        if word.lower() not in stop
    ]
    for sent in train.text.values
]



In [4]:
NUM_FEATURES = 200

# Fix: gensim>=4 uses vector_size instead of size.
model = Word2Vec(
    sentences=data,
    min_count=3,
    vector_size=NUM_FEATURES,
    window=5,
    sg=1,
    alpha=1e-4,
    workers=4,
)



In [5]:
# Fix: gensim 4 removed .vocab; use index_to_key
len(model.wv.index_to_key)




10253

In [6]:
# Fix: gensim 4 access vectors via model.wv; keep same semantics otherwise.
def get_feature_vec(tokens, num_features, model):
    featureVec = np.zeros(shape=(1, num_features), dtype="float32")
    missed = 0
    for word in tokens:
        try:
            featureVec = np.add(featureVec, model.wv[word])
        except KeyError:
            missed += 1
            pass
    if len(tokens) - missed == 0:
        return np.zeros(shape=(num_features,), dtype="float32")
    return np.divide(featureVec, len(tokens) - missed).squeeze()




In [7]:
vectors = []
for i in train.text.values:
    tokens = [
        lemmatizer.lemmatize(word.lower())
        for word in alpha_tokenizer.tokenize(i)
        if word.lower() not in stop
    ]
    vectors.append(get_feature_vec(tokens, NUM_FEATURES, model))

X_train = np.vstack(vectors).astype(np.float32)



In [8]:
train["author"] = train["author"].map({"EAP": 0, "HPL": 1, "MWS": 2})
y_train = train.author.values



In [9]:
# Fix: ensure model converges reliably (prevents warnings/errors); core model unchanged.
estimator = LogisticRegression(C=1, max_iter=1000, multi_class="auto")
estimator.fit(X_train, y_train)



LogisticRegression(C=1, max_iter=1000)

In [10]:
test_vectors = []
for i in test.text.values:
    tokens = [
        lemmatizer.lemmatize(word.lower())
        for word in alpha_tokenizer.tokenize(i)
        if word.lower() not in stop
    ]
    test_vectors.append(get_feature_vec(tokens, NUM_FEATURES, model))

X_test = np.vstack(test_vectors).astype(np.float32)



In [11]:
probs = estimator.predict_proba(X_test)



In [12]:
# Fix: ensure correct submission column order and valid CSV output.
author = pd.DataFrame(probs, columns=["EAP", "HPL", "MWS"])

final = pd.DataFrame()
final["id"] = test.id.values
final["EAP"] = author["EAP"].values
final["HPL"] = author["HPL"].values
final["MWS"] = author["MWS"].values

final.to_csv("submission.csv", sep=",", index=False)
print(final.head())
print("Wrote submission.csv with shape:", final.shape)

        id       EAP       HPL       MWS
0  id27251  0.401665  0.288349  0.309987
1  id09612  0.403096  0.287645  0.309259
2  id11943  0.401324  0.287589  0.311087
3  id19526  0.404039  0.289221  0.306740
4  id12931  0.402197  0.287178  0.310625
Wrote submission.csv with shape: (1958, 4)
